# XGBoost Pipeline — Trajectory Prediction Metrics (GAM-Style)

This notebook fits an XGBoost regressor to trajectory prediction error metrics using prepared trajectory/scene characteristics.
It mirrors the structure of `gam.ipynb`, uses a random train/validation/test split, and focuses interpretability on conditions where performance is poor vs. well.

## 1. Imports and Configuration

In [1]:
from pathlib import Path
import sys

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    src_dir = candidate / 'src'
    if (src_dir / 'data_modelling').exists():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break
else:
    raise RuntimeError('Could not locate repo src/ directory for notebook imports.')


In [2]:
# Core libraries
import warnings
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import xgboost as xgb
import optuna

from data_modelling.common_metrics import to_original_scale
from data_modelling.prepared_data import load_prepared_data, prepare_single_target_model_data
from data_modelling.training_outputs import (
    build_oof_frame,
    build_oof_metrics_df,
    build_run_manifest,
    summarize_nested_cv,
    write_manifest,
)

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Notebook-level config constants
MODEL_ID = 'xgboost'
RUN_NAME = "nusc_mini_debug_tpp-11_Mar_2026_15_29_02"
PREPARED_TARGET_COL = "ml_ade"
DATA_PATH = Path('../../results/interpretable_model/prepared_data') / RUN_NAME / f"prepared_data_{PREPARED_TARGET_COL}.csv"
TARGET_COL = None  # Optional override, e.g. 'ml_ade_log'

RANDOM_STATE = 42
k_outer_fold = 5
k_inner_fold = 3
N_OPTUNA_TRIALS = 40
MAX_BOOST_ROUNDS = 2000
EARLY_STOPPING_ROUNDS = 50
N_JOBS = -1
POOR_WELL_QUANTILE = 0.20
TUNING_METRIC = 'rmse'

SAVE_DIR = Path('../../results/interpretable_model') / MODEL_ID / RUN_NAME
PLOTS_DIR = SAVE_DIR / 'plots'
TABLES_DIR = SAVE_DIR / 'tables'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print('Imports and configuration loaded.')
print(f'Run: {RUN_NAME}')
print(f'Model ID: {MODEL_ID}')
print('Interpretable model: XGBoost')
print(f'DATA_PATH: {DATA_PATH}')
print(f'SAVE_DIR:  {SAVE_DIR.resolve()}')
print(
    'Nested CV: '
    f'outer={k_outer_fold}, inner={k_inner_fold}, optuna_trials={N_OPTUNA_TRIALS}, '
    f'early_stop={EARLY_STOPPING_ROUNDS}'
)


Imports and configuration loaded.
Run: nusc_mini_debug_tpp-11_Mar_2026_15_29_02
Model ID: xgboost
Interpretable model: XGBoost
DATA_PATH: ../../results/interpretable_model/prepared_data/nusc_mini_debug_tpp-11_Mar_2026_15_29_02/prepared_data_ml_ade.csv
SAVE_DIR:  /Users/simondrauz/Lokale Dokumente/Repositories/ds_practical/results/interpretable_model/xgboost/nusc_mini_debug_tpp-11_Mar_2026_15_29_02
Nested CV: outer=5, inner=3, optuna_trials=40, early_stop=50


## 2. Load Prepared Data and Inspect

In [3]:
df = load_prepared_data(
    DATA_PATH,
    display_fn=display,
    include_missing_summary=True,
    include_dtype_summary=True,
)


Dataset shape: (451, 14)
Columns:
['max_speed', 'std_speed', 'mean_acceleration', 'mean_jerk', 'heading_change', 'has_collision', 'min_neighbor_distance', 'scene_num_agents', 'scene_bbox_area', 'scene_bbox_width', 'scene_bbox_height', 'scene_spatial_density', 'scene_density_VEHICLE', 'ml_ade_log']


,max_speed,std_speed,mean_acceleration,mean_jerk,heading_change,has_collision,min_neighbor_distance,scene_num_agents,scene_bbox_area,scene_bbox_width,scene_bbox_height,scene_spatial_density,scene_density_VEHICLE,ml_ade_log
0,1.230403,0.116722,0.056417,0.490805,112.440745,0.0,1.013903,17.0,1160.377375,40.869190,28.392473,0.014650,0.005171,0.649643
1,0.218477,0.063105,0.007725,0.490449,941.304051,0.0,3.023524,17.0,1160.377375,40.869190,28.392473,0.014650,0.005171,0.140704
2,0.383437,0.099171,0.031384,0.401269,488.173829,0.0,1.104355,50.0,1821.083675,44.533135,40.892780,0.027456,0.012630,0.388765
3,1.483624,0.190256,0.051802,0.879107,81.155053,1.0,0.426945,16.0,1230.401053,43.391608,28.355738,0.013004,0.005689,0.821990
4,0.218477,0.064457,0.010995,0.483671,946.683531,0.0,2.796330,15.0,1057.088774,37.340273,28.309616,0.014190,0.005676,0.159340



Missing values per column:


,missing_count
max_speed,0
std_speed,0
mean_acceleration,0
mean_jerk,0
heading_change,0
has_collision,0
min_neighbor_distance,0
scene_num_agents,0
scene_bbox_area,0
scene_bbox_width,0



Column dtypes:


,dtype
max_speed,float64
std_speed,float64
mean_acceleration,float64
mean_jerk,float64
heading_change,float64
has_collision,float64
min_neighbor_distance,float64
scene_num_agents,float64
scene_bbox_area,float64
scene_bbox_width,float64


## 3. Resolve Target and Build Feature Matrix

In [4]:
prepared = prepare_single_target_model_data(
    df,
    target_col=TARGET_COL,
    default_target='ml_ade',
)
target_col = prepared['target_col']
feature_cols = prepared['feature_cols']
model_df = prepared['model_df']
X = prepared['X']
y = prepared['y']
row_ids = prepared['row_ids']

print(f'Target column: {target_col}')
print(f'Number of features: {len(feature_cols)}')
print(f'Rows available for modeling: {len(model_df)}')
print(f'Feature matrix shape: {X.shape}')
print(f'Target vector shape: {y.shape}')


Target column: ml_ade_log
Number of features: 13
Rows available for modeling: 451
Feature matrix shape: (451, 13)
Target vector shape: (451,)


## 4. Outer/Inner CV Setup (OOF + Optuna)


In [5]:
outer_cv = KFold(n_splits=k_outer_fold, shuffle=True, random_state=RANDOM_STATE)

n_samples = len(model_df)
oof_pred = np.full(n_samples, np.nan, dtype=float)
oof_fold = np.full(n_samples, -1, dtype=int)

print(f'Outer CV folds: {k_outer_fold}')
print(f'Inner CV folds: {k_inner_fold}')
print(f'Rows for OOF prediction: {n_samples}')


Outer CV folds: 5
Inner CV folds: 3
Rows for OOF prediction: 451


## 5. Nested Resampling + Optuna Hyperparameter Optimization (xgb.cv)


In [6]:
base_params = {
    'objective': 'reg:squarederror',
    'eval_metric': TUNING_METRIC,
    'tree_method': 'hist',
    'verbosity': 0,
    'nthread': N_JOBS,
}


def run_optuna_tuning(X_train, y_train, *, seed, tuning_scope):
    dtrain = xgb.DMatrix(X_train, label=y_train)
    trial_rows = []

    def objective(trial):
        params = base_params.copy()
        params.update({
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-6, 20.0, log=True),
            'seed': seed,
        })

        cv_results = xgb.cv(
            params=params,
            dtrain=dtrain,
            nfold=k_inner_fold,
            num_boost_round=MAX_BOOST_ROUNDS,
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            metrics=TUNING_METRIC,
            seed=seed,
            shuffle=True,
            verbose_eval=False,
        )

        best_iteration = int(cv_results[f'test-{TUNING_METRIC}-mean'].idxmin() + 1)
        best_cv_score = float(cv_results[f'test-{TUNING_METRIC}-mean'].min())
        trial.set_user_attr('best_iteration', best_iteration)
        trial.set_user_attr('best_cv_score', best_cv_score)

        row = {
            'tuning_scope': tuning_scope,
            'trial_number': trial.number,
            'best_iteration': best_iteration,
            'best_cv_score': best_cv_score,
        }
        row.update({k: v for k, v in params.items() if k not in base_params})
        trial_rows.append(row)
        return best_cv_score

    study = optuna.create_study(
        direction='minimize',
        sampler=optuna.samplers.TPESampler(seed=seed),
    )
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=False)

    trial_results_df = pd.DataFrame(trial_rows).sort_values('best_cv_score').reset_index(drop=True)
    best_iteration = int(study.best_trial.user_attrs['best_iteration'])
    best_cv_score = float(study.best_trial.user_attrs['best_cv_score'])

    print(
        f'{tuning_scope} tuning complete | '
        f'best_{TUNING_METRIC}={best_cv_score:.6f} | '
        f'best_iteration={best_iteration}'
    )

    return {
        'best_params': study.best_params,
        'best_iteration': best_iteration,
        'best_cv_score': best_cv_score,
        'trial_results_df': trial_results_df,
    }


nested_rows = []

for fold_idx, (outer_train_idx, outer_valid_idx) in enumerate(outer_cv.split(X, y), start=1):
    X_outer_train = X.iloc[outer_train_idx]
    y_outer_train = y.iloc[outer_train_idx]
    X_outer_valid = X.iloc[outer_valid_idx]
    y_outer_valid = y.iloc[outer_valid_idx]

    tuning_result = run_optuna_tuning(
        X_outer_train,
        y_outer_train,
        seed=RANDOM_STATE + fold_idx,
        tuning_scope=f'outer_fold_{fold_idx}',
    )

    train_params = base_params.copy()
    train_params.update(tuning_result['best_params'])
    train_params['seed'] = RANDOM_STATE + fold_idx

    dtrain = xgb.DMatrix(X_outer_train, label=y_outer_train)
    dvalid = xgb.DMatrix(X_outer_valid, label=y_outer_valid)
    booster = xgb.train(train_params, dtrain, num_boost_round=tuning_result['best_iteration'])
    oof_pred[outer_valid_idx] = booster.predict(dvalid)
    oof_fold[outer_valid_idx] = fold_idx

    y_outer_valid_orig = to_original_scale(y_outer_valid, target_col=target_col)
    outer_pred_orig = to_original_scale(oof_pred[outer_valid_idx], target_col=target_col)

    outer_rmse = float(np.sqrt(mean_squared_error(y_outer_valid_orig, outer_pred_orig)))
    outer_mae = mean_absolute_error(y_outer_valid_orig, outer_pred_orig)
    outer_r2 = r2_score(y_outer_valid_orig, outer_pred_orig)

    row = {
        'outer_fold': fold_idx,
        'outer_rmse': outer_rmse,
        'outer_mae': outer_mae,
        'outer_r2': outer_r2,
        'inner_best_cv_rmse': tuning_result['best_cv_score'],
        'best_iteration': tuning_result['best_iteration'],
    }
    row.update({f'best_{k}': v for k, v in tuning_result['best_params'].items()})
    nested_rows.append(row)

    print(
        f'Outer fold {fold_idx}/{k_outer_fold} | '
        f'RMSE={outer_rmse:.6f} | MAE={outer_mae:.6f} | R2={outer_r2:.4f}'
    )

if np.isnan(oof_pred).any():
    raise ValueError('OOF predictions contain NaN values. Check CV splits and training flow.')

nested_cv_df = pd.DataFrame(nested_rows)
display(nested_cv_df)

nested_summary = summarize_nested_cv(nested_cv_df)
print('Nested CV summary:')
display(nested_summary)

nested_path = TABLES_DIR / f'nested_cv_optuna_{target_col}.csv'
nested_summary_path = TABLES_DIR / f'nested_cv_optuna_summary_{target_col}.csv'
nested_cv_df.to_csv(nested_path, index=False)
nested_summary.to_csv(nested_summary_path, index=False)
print(f'Nested CV fold results saved to: {nested_path}')
print(f'Nested CV summary saved to:      {nested_summary_path}')

# Build OOF prediction table
model_df_oof = build_oof_frame(
    model_df,
    row_ids=row_ids,
    oof_pred=oof_pred,
    oof_fold=oof_fold,
    target_orig=to_original_scale(model_df[target_col].values, target_col=target_col),
    pred_scale_kwargs={'target_col': target_col},
)

oof_path = TABLES_DIR / f'oof_predictions_{target_col}.csv'
model_df_oof[['row_id', 'oof_pred', 'oof_pred_orig', 'target_orig', 'outer_fold']].to_csv(oof_path, index=False)
print(f'OOF predictions saved to: {oof_path}')

full_data_tuning = run_optuna_tuning(
    X,
    y,
    seed=RANDOM_STATE,
    tuning_scope='full_data',
)
full_data_tuning_trials_path = TABLES_DIR / f'full_data_tuning_optuna_trials_{target_col}.csv'
full_data_tuning['trial_results_df'].to_csv(full_data_tuning_trials_path, index=False)

full_data_tuning_summary = {
    'model_id': MODEL_ID,
    'run_name': RUN_NAME,
    'target_col': target_col,
    'tuning_metric_name': TUNING_METRIC,
    'best_params': full_data_tuning['best_params'],
    'best_iteration': full_data_tuning['best_iteration'],
    'best_cv_score': full_data_tuning['best_cv_score'],
    'tuning_config': {
        'nfold': k_inner_fold,
        'n_trials': N_OPTUNA_TRIALS,
        'max_boost_rounds': MAX_BOOST_ROUNDS,
        'early_stopping_rounds': EARLY_STOPPING_ROUNDS,
        'random_state': RANDOM_STATE,
    },
}
full_data_tuning_summary_path = TABLES_DIR / f'full_data_tuning_optuna_summary_{target_col}.json'
full_data_tuning_summary_path.write_text(json.dumps(full_data_tuning_summary, indent=2))

print('Selected hyperparameters from full-data retuning:')
print(full_data_tuning['best_params'])
print(f"Best iteration from full-data retuning: {full_data_tuning['best_iteration']}")
print(f'Full-data tuning trials saved to: {full_data_tuning_trials_path}')
print(f'Full-data tuning summary saved to: {full_data_tuning_summary_path}')

model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=full_data_tuning['best_iteration'],
    tree_method='hist',
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
    verbosity=0,
    **full_data_tuning['best_params'],
)
model.fit(X, y, verbose=False)
print('Final model fitted on full data using retuned hyperparameters.')


outer_fold_1 tuning complete | best_rmse=0.149422 | best_iteration=769
Outer fold 1/5 | RMSE=0.279867 | MAE=0.196949 | R2=0.7495
outer_fold_2 tuning complete | best_rmse=0.147910 | best_iteration=1987
Outer fold 2/5 | RMSE=0.300870 | MAE=0.183387 | R2=0.7410
outer_fold_3 tuning complete | best_rmse=0.154658 | best_iteration=862
Outer fold 3/5 | RMSE=0.255109 | MAE=0.174916 | R2=0.8242
outer_fold_4 tuning complete | best_rmse=0.169710 | best_iteration=1506
Outer fold 4/5 | RMSE=0.277839 | MAE=0.199136 | R2=0.6919
outer_fold_5 tuning complete | best_rmse=0.153299 | best_iteration=1746
Outer fold 5/5 | RMSE=0.262828 | MAE=0.186164 | R2=0.8266


,outer_fold,outer_rmse,outer_mae,outer_r2,inner_best_cv_rmse,best_iteration,best_learning_rate,best_max_depth,best_min_child_weight,best_subsample,best_colsample_bytree,best_reg_alpha,best_reg_lambda
0,1,0.279867,0.196949,0.749544,0.149422,769,0.032016,8,12,0.879914,0.864987,0.002043,0.000001
1,2,0.300870,0.183387,0.741013,0.147910,1987,0.014484,10,4,0.683135,0.859981,0.001159,0.000825
2,3,0.255109,0.174916,0.824240,0.154658,862,0.037512,9,5,0.999098,0.893826,0.005536,0.001120
3,4,0.277839,0.199136,0.691859,0.169710,1506,0.013080,9,3,0.701835,0.877413,0.000015,4.839413
4,5,0.262828,0.186164,0.826571,0.153299,1746,0.029138,10,16,0.937803,0.975063,0.000001,0.406650


Nested CV summary:


,metric,mean,std
0,outer_rmse,0.275303,0.017638
1,outer_mae,0.188110,0.009999
2,outer_r2,0.766646,0.057989


Nested CV fold results saved to: ../../results/interpretable_model/xgboost/nusc_mini_debug_tpp-11_Mar_2026_15_29_02/tables/nested_cv_optuna_ml_ade_log.csv
Nested CV summary saved to:      ../../results/interpretable_model/xgboost/nusc_mini_debug_tpp-11_Mar_2026_15_29_02/tables/nested_cv_optuna_summary_ml_ade_log.csv
OOF predictions saved to: ../../results/interpretable_model/xgboost/nusc_mini_debug_tpp-11_Mar_2026_15_29_02/tables/oof_predictions_ml_ade_log.csv
full_data tuning complete | best_rmse=0.136525 | best_iteration=1980
Selected hyperparameters from full-data retuning:
{'learning_rate': 0.046589863196059025, 'max_depth': 8, 'min_child_weight': 20, 'subsample': 0.9775699684882246, 'colsample_bytree': 0.8259332753890892, 'reg_alpha': 0.0015027010896820315, 'reg_lambda': 0.21499611376334174}
Best iteration from full-data retuning: 1980
Full-data tuning trials saved to: ../../results/interpretable_model/xgboost/nusc_mini_debug_tpp-11_Mar_2026_15_29_02/tables/full_data_tuning_optuna

## 6. Evaluate on OOF Predictions (Original Scale)


In [7]:
metrics_df = build_oof_metrics_df(y, oof_pred, target_col=target_col)
display(metrics_df)

metrics_path = TABLES_DIR / f'metrics_oof_{target_col}.csv'
metrics_df.to_csv(metrics_path, index=False)
print(f'OOF metrics saved to: {metrics_path}')


,Split,R²,MAE,RMSE
0,OOF,0.775424,0.18813,0.275763


OOF metrics saved to: ../../results/interpretable_model/xgboost/nusc_mini_debug_tpp-11_Mar_2026_15_29_02/tables/metrics_oof_ml_ade_log.csv


## 7. Export Artifacts and Run Manifest


In [8]:
# Persist model for downstream analysis notebooks
model_path = SAVE_DIR / f'xgb_model_{target_col}.json'
model.save_model(model_path)

# Persist modeling data with OOF predictions
data_path = TABLES_DIR / f'model_data_with_oof_{target_col}.csv'
model_df_oof.to_csv(data_path, index=False)

manifest = build_run_manifest(
    model_id=MODEL_ID,
    run_name=RUN_NAME,
    target_col=target_col,
    feature_cols=feature_cols,
    save_dir=SAVE_DIR,
    plots_dir=PLOTS_DIR,
    tables_dir=TABLES_DIR,
    nested_resampling={
        'nested_cv_path': str(nested_path),
        'nested_cv_summary_path': str(nested_summary_path),
        'oof_predictions_path': str(oof_path),
        'oof_metrics_path': str(metrics_path),
        'model_data_with_oof_path': str(data_path),
    },
    final_model={
        'model_path': str(model_path),
        'full_data_tuning_trials_path': str(full_data_tuning_trials_path),
        'full_data_tuning_summary_path': str(full_data_tuning_summary_path),
        'selected_best_params': full_data_tuning['best_params'],
        'selected_best_iteration': full_data_tuning['best_iteration'],
        'best_cv_score': full_data_tuning['best_cv_score'],
        'tuning_metric_name': TUNING_METRIC,
    },
    analysis={
        'poor_well_quantile': POOR_WELL_QUANTILE,
    },
)
manifest_path = write_manifest(manifest, TABLES_DIR, target_col)

print(f'Interpretable model saved to: {model_path}')
print(f'Model data saved to: {data_path}')
print(f'Run manifest saved to: {manifest_path}')


Interpretable model saved to: ../../results/interpretable_model/xgboost/nusc_mini_debug_tpp-11_Mar_2026_15_29_02/xgb_model_ml_ade_log.json
Model data saved to: ../../results/interpretable_model/xgboost/nusc_mini_debug_tpp-11_Mar_2026_15_29_02/tables/model_data_with_oof_ml_ade_log.csv
Run manifest saved to: ../../results/interpretable_model/xgboost/nusc_mini_debug_tpp-11_Mar_2026_15_29_02/tables/run_manifest_ml_ade_log.json
